In [1]:
import math
from collections import Counter

In [2]:
def entropy(labels):
    total = len(labels)
    counts = Counter(labels)

    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)
    return ent

In [3]:
def information_gain(data, labels, feature_index):
    total_entropy = entropy(labels)
    values = set(row[feature_index] for row in data)

    weighted_entropy = 0
    for v in values:
        sub_labels = [
            labels[i] for i in range(len(data)) if data[i][feature_index] == v
        ]
        weighted_entropy += (len(sub_labels) / len(labels)) * entropy(sub_labels)

    return total_entropy - weighted_entropy

In [4]:
def build_tree(data, labels, features):
    if len(set(labels)) == 1:
        return labels[0]

    if not features:
        return Counter(labels).most_common(1)[0][0]

    gains = {
        f: information_gain(data, labels, f) for f in features
    }
    best_feature = max(gains, key=gains.get)

    tree = {best_feature: {}}

    feature_values = set(row[best_feature] for row in data)

    for value in feature_values:
        sub_data = []
        sub_labels = []

        for i in range(len(data)):
            if data[i][best_feature] == value:
                sub_data.append(data[i])
                sub_labels.append(labels[i])

        remaining_features = [f for f in features if f != best_feature]
        tree[best_feature][value] = build_tree(
            sub_data, sub_labels, remaining_features
        )

    return tree

In [5]:
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample[feature]

    if value in tree[feature]:
        return predict(tree[feature][value], sample)
    else:
        return None

In [6]:
# 0: Weather, 1: Temperature

X = [
    ["Sunny", "Hot"],
    ["Sunny", "Hot"],
    ["Overcast", "Hot"],
    ["Rain", "Mild"],
    ["Rain", "Cool"],
    ["Rain", "Cool"],
    ["Overcast", "Cool"],
    ["Sunny", "Mild"]
]

y = ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No"]

features = [0, 1]

tree = build_tree(X, y, features)
print("Decision Tree:", tree)

# Test
sample = ["Sunny", "Cool"]
print("Prediction:", predict(tree, sample))

Decision Tree: {0: {'Overcast': 'Yes', 'Sunny': 'No', 'Rain': {1: {'Mild': 'Yes', 'Cool': 'Yes'}}}}
Prediction: No
